In [1]:
# # Evandro Ribeiro Gomes Coelho
# # TCCII

# packagess
import pandas as pd
import numpy as np

In [2]:
# # import data

fl_path = 'data/'
fl_ext = '.csv'

fl_name = 'plfec-holidays'
df_holidays = pd.DataFrame(pd.read_csv(fl_path + fl_name + fl_ext, sep=';'))

fl_name = 'plfec-metadata'
df_metadata = pd.DataFrame(pd.read_csv(fl_path + fl_name + fl_ext, sep=';'))

fl_name = 'plfec-submission-forecast-period'
df_submission_forecast_period = pd.DataFrame(pd.read_csv(fl_path + fl_name + fl_ext, sep=';'))

fl_name = 'plfec-test-data'
df_test = pd.DataFrame(pd.read_csv(fl_path + fl_name + fl_ext, sep=';'))

fl_name = 'plfec-train-data'
df_train = pd.DataFrame(pd.read_csv(fl_path + fl_name + fl_ext, sep=';'))

fl_name = 'plfec-weather'
df_weather = pd.DataFrame(pd.read_csv(fl_path + fl_name + fl_ext, sep=';'))

# delete temp variables
del fl_name, fl_path, fl_ext

In [24]:
df_holidays[df_holidays['SiteId'] == 40]

,Date,SiteId,Holiday Flag
1425,2015-01-01,40,1
1808,2015-04-06,40,1
2105,2015-05-01,40,1
2261,2015-05-08,40,1
2276,2015-05-14,40,1
2309,2015-05-25,40,1
2487,2015-07-14,40,1
2564,2015-08-15,40,1
2736,2015-11-01,40,1
2864,2015-11-11,40,1


In [4]:
df_metadata['SiteId'].count()

267

In [5]:
# df_holiday
df_holidays = df_holidays.groupby(['Date', 'SiteId']).agg({'Holiday':'count'}).reset_index().rename(columns={'Holiday': 'Holiday Flag'})
df_holidays['Holiday Flag'] = int(1)
df_holidays = df_holidays.sort_values(by=['SiteId', 'Date'])

In [6]:
# triming the data
df_train['Date'] = pd.to_datetime(df_train['Timestamp'].str[:10])
df_test['Date'] = pd.to_datetime(df_test['Timestamp'].str[:10])
df_holidays['Date'] = pd.to_datetime(df_holidays['Date'])

In [7]:
# converting Timestamp to datetime
df_train['Timestamp'] = pd.to_datetime(df_train['Timestamp'].str[:10] + " " + df_train['Timestamp'].str[11:19])
df_test['Timestamp'] = pd.to_datetime(df_test['Timestamp'].str[:10] + " " + df_test['Timestamp'].str[11:19])
df_weather['Timestamp'] = pd.to_datetime(df_weather['Timestamp'].str[:10] + " " + df_weather['Timestamp'].str[11:19])

In [8]:
# tag each group
df_test['tag'] = 'test'
df_train['tag'] = 'train'

In [9]:
# merge train and test dataframes

df_energy = pd.merge(
    df_test,
    df_train,
    how="outer",
    on=None,
)

In [10]:
# sort df_energy by Timestamp

df_energy = df_energy.sort_values(by=['SiteId','Timestamp'])

In [11]:
# get the closest weather value by distance in df_weather

# há mais de uma medida de Temperature para o mesmo SiteId na mesma Timestamp, de distâncias diferentes. Utilizado o valor da menor distância.
df_weather = df_weather.sort_values('Distance').drop_duplicates(subset=['SiteId', 'Timestamp'], keep='first')
df_weather = df_weather.sort_values(by=['Timestamp'])

In [12]:
# energy frequence of measure (minutes)
df_energy['freq_energy'] = df_energy['Timestamp'].diff().apply(lambda x: x/np.timedelta64(1, 'm')).fillna(0).astype('int64')

# weather frequence of measure (minutes)
df_weather['freq_weather'] = df_weather['Timestamp'].diff().apply(lambda x: x/np.timedelta64(1, 'm')).fillna(0).astype('int64')

In [13]:
# merge df with df_weather

df_merge1 = pd.merge(
    df_energy, # left
    df_weather, # right
    how="left",
    on=['SiteId', 'Timestamp'],
)

In [14]:
# 397634
df_weather.count()

Timestamp       2708318
Temperature     2708318
Distance        2708318
SiteId          2708318
freq_weather    2708318
dtype: int64

In [15]:
# merge df with df_holidays

df_merge2 = pd.merge(
    df_merge1,
    df_holidays,
    how="left",
    on=['Date', 'SiteId'],
)

In [16]:
df_merge2.count()

obs_id          7869006
SiteId          7869006
Timestamp       7869006
ForecastId      7869006
Value           7763343
Date            7869006
tag             7869006
freq_energy     7869006
Temperature      397634
Distance         397634
freq_weather     397634
Holiday Flag     182842
dtype: int64

In [17]:
# merge df with df_metadata

df_merge3 = pd.merge(
    df_merge2, # left
    df_metadata, # right
    how="left",
    on=['SiteId'],
)

In [18]:
# merge df with df_submission_forecast_period

df_merge4 = pd.merge(
    df_merge3, # left
    df_submission_forecast_period, # right
    how="left",
    on=['ForecastId'],
)

In [19]:
# remane dfs and delete the aux merges

df = df_merge4
del df_merge1, df_merge2, df_merge3, df_merge4

In [20]:
df.count()

obs_id               7869006
SiteId               7869006
Timestamp            7869006
ForecastId           7869006
Value                7763343
Date                 7869006
tag                  7869006
freq_energy          7869006
Temperature           397634
Distance              397634
freq_weather          397634
Holiday Flag          182842
Surface              7869006
Sampling             7869006
BaseTemperature      7869006
MondayIsDayOff       7869006
TuesdayIsDayOff      7869006
WednesdayIsDayOff    7869006
ThursdayIsDayOff     7869006
FridayIsDayOff       7869006
SaturdayIsDayOff     7869006
SundayIsDayOff       7869006
ForecastPeriodNS     7869006
dtype: int64

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7869006 entries, 0 to 7869005
Data columns (total 23 columns):
 #   Column             Dtype         
---  ------             -----         
 0   obs_id             int64         
 1   SiteId             int64         
 2   Timestamp          datetime64[ns]
 3   ForecastId         int64         
 4   Value              float64       
 5   Date               datetime64[ns]
 6   tag                object        
 7   freq_energy        int64         
 8   Temperature        float64       
 9   Distance           float64       
 10  freq_weather       float64       
 11  Holiday Flag       float64       
 12  Surface            float64       
 13  Sampling           float64       
 14  BaseTemperature    float64       
 15  MondayIsDayOff     bool          
 16  TuesdayIsDayOff    bool          
 17  WednesdayIsDayOff  bool          
 18  ThursdayIsDayOff   bool          
 19  FridayIsDayOff     bool          
 20  SaturdayIsDayOff   bool 

In [23]:
# saving data to csv

df.to_csv('data/df_full.csv', index = False, header=True)